# CTU-13 — Dataset Exploration

Botnet traffic across 13 capture scenarios (7 distinct botnets).

**Files:** 13 Parquet files, one per scenario, named `{id}-{BotnetFamily}-{date}.binetflow.parquet`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
BASE = Path('../../dataset/ctu13/')

## 1 · Load All Scenarios

In [ ]:
files = sorted(BASE.glob('*.parquet'))
print(f'Parquet files found: {len(files)}')
for f in files:
    print(f'  {f.name}')

In [ ]:
dfs = []
for f in files:
    tmp = pd.read_parquet(f)
    tmp['scenario'] = f.stem.split('-')[0]          # numeric scenario id
    tmp['botnet']   = f.stem.split('-')[1]          # botnet family name
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'Combined shape: {df.shape}')
df.head(3)

## 2 · Basic Inspection

In [ ]:
print('Column dtypes:')
print(df.dtypes)
print()
print('Sample label values:')
print(df['Label'].value_counts().head(20).to_string() if 'Label' in df.columns
      else 'No Label column — check column names above')

## 3 · Missing Values

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0].to_string() or 'No missing values')

## 4 · Label Distribution

In [ ]:
label_col = [c for c in df.columns if 'label' in c.lower() or 'Label' in c][0]
print(f'Label column: "{label_col}"')

# Binary: Background/Normal vs Botnet
df['is_botnet'] = (~df[label_col].str.contains('Background|LEGITIMATE', case=False, na=False)).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[label_col].value_counts().head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 raw labels')
df['is_botnet'].map({0:'Background/Normal', 1:'Botnet'}).value_counts()\
    .plot(kind='bar', ax=axes[1], color=['steelblue','tomato'])
axes[1].set_title('Binary split')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 5 · Distribution per Scenario

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
df.groupby('scenario')['is_botnet'].value_counts().unstack().plot(
    kind='bar', stacked=True, ax=ax, color=['steelblue','tomato'])
ax.set_title('Botnet vs background per capture scenario')
ax.set_xlabel('Scenario ID'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 6 · Feature Statistics

In [ ]:
num_cols = df.select_dtypes('number').drop(columns=['is_botnet','scenario'], errors='ignore').columns
df[num_cols].describe().T.round(3)

## 7 · Preprocessing Checklist

| Step | Required | Notes |
|---|---|---|
| Parse label to binary or multi-class | Yes | 'Background' = 0, rest = 1 (binary) |
| Encode categorical features | Yes | Protocol, flags if present |
| Scale numerics | Yes | Bytes/duration are heavily skewed |
| Handle missing values | Check | Some NetFlow fields may be NaN |
| Scenario leakage | Yes | Do NOT train and test on same scenario |
| Class imbalance | Severe | Background traffic dominates every scenario |

## 8 · Train / Test Split

**Important:** split by scenario, not by row, to avoid data leakage (flows from the same capture period are temporally correlated).

In [ ]:
scenarios = sorted(df['scenario'].unique())
print(f'Scenarios: {scenarios}')

# Hold out last 3 scenarios as test (unseen botnet captures)
test_scenarios  = scenarios[-3:]
train_scenarios = scenarios[:-3]

train_df = df[df['scenario'].isin(train_scenarios)]
test_df  = df[df['scenario'].isin(test_scenarios)]

print(f'Train scenarios {train_scenarios}: {train_df.shape}')
print(f'Test  scenarios {test_scenarios}:  {test_df.shape}')
print(f'Train botnet rate: {train_df["is_botnet"].mean():.2%}')
print(f'Test  botnet rate: {test_df["is_botnet"].mean():.2%}')

## 9 · Summary

- **Size:** 13 scenarios, combined ~several million flows (varies by scenario)
- **Features:** NetFlow-derived (duration, bytes, packets, protocol, flags)
- **Labels:** Background / botnet family; typically treated as binary
- **Key issues:** Extreme class imbalance, temporal correlation within scenarios
- **Recommended split:** By scenario index — never random row shuffle